# Multimer analysis VI - external annotation
Here I take the original data from PB and CSF, and "deorphanize" as many TCRs as possible using the multimer data as annotations along with computational methods as additional annotations

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
import scvi
import torch

# from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
import scipy.stats as stats
# import string

# Helps to define plotting area

import matplotlib.pyplot as plt
import seaborn.objects as so
from scipy.stats import kruskal
import scikit_posthocs as sp
from skbio.diversity.alpha import gini_index, shannon, simpson
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#Set seaborn theme
sb.set_style("ticks")
# custom_params = {"axes.spines.right": False, "axes.spines.top": False}
# sb.set_theme(style="ticks", rc=custom_params)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

# Loading CD8 T cell data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")
len(mdata)

# Annotate using clone assignments from the multimer data

In [ ]:
mult_ann = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/clone_x_multimer_assignments.xlsx")
mult_ann = mult_ann[~mult_ann["clone_handle"].isna()]
mult_ann = mult_ann[~mult_ann["handle"].isna()]
mult_ann = mult_ann[["clone_handle", "handle", "Virus", "Antigen", "HLA"]]
mult_ann = mult_ann.drop_duplicates()

# # Return a list of antigens
# ag_list = mult_ann["Antigen"].drop_duplicates().tolist()
# ag_list

mult_ann.head(5)

# Annotate using ImmuneWatch predictions (primary chains)

In [ ]:
IMW_ann = pd.read_table(sep="\t", filepath_or_buffer="../../03_annotation/12_computational_agspecificity_annotation/Immunewatch_1.4.7/2027_7_full_CD8T_mults_main_predictions.tsv")
print(IMW_ann.columns) 
#Cutoff suggestion for the Score is 0.2
IMW_ann = IMW_ann[IMW_ann["Score"] >= 0.2]

#select columns
IMW_ann = IMW_ann[["clone_id", "Epitope", "Epitope Antigen", "Epitope Species", "Epitope MHC"]]

## Translate epitope species 
antigen_map = {
    "Phosphoprotein 65-Mutant (pp65-Mutant)": "pp65",
    "Phosphoprotein 65 (pp65)": "pp65",
    "Matrix protein 1 (M1)": "M",  # influenza matrix protein
    "Epstein-Barr nuclear antigen 1 (EBNA1)": "EBNA1",
    "Spike/surface glycoprotein (S)": "S",
    "Epstein-Barr nuclear antigen 3A (EBNA3A),Epstein-Barr nuclear antigen 3 (EBNA3)": "EBNA3",
    "mRNA export factor ICP27 homolog (BMLF1),Replication and transcription activator (BRLF1)": "BMLF1",
    "NS3-Mutant,K1Y": "NS3mut",
    "p24 Gag": "p24",
    "Epstein-Barr nuclear antigen 3A (EBNA3A),Epstein-Barr nuclear antigen 3 (EBNA3),Trans activator protein (BZLF1)": "EBNA3",
    "Latent membrane protein 2A (LMP2A),Latent membrane protein 2 (LMP2)": "LMP2",
    "Gag,p24 Gag": "p24",
    "Transient Receptor Potential Cation Channel Subfamily V Member 4-Mutant (TRPV4-Mutant)": "TRPV4mut",
    "Non-structural protein 4b (NS4b),Polyprotein": "NS4b",
    "Splicing factor 3B subunit 1-Mutant (SF3B1-Mutant)": "SF3B1mut",
    "KRAS-Mutant": "KRASmut",
    "Nucleocapsid protein (N)": "N",
    "Trans activator protein (BZLF1)": "BZLF1",
    "Melanoma antigen recognized by T-cells 1 (MART1)": "MART1",
    "Epstein-Barr nuclear antigen 3A (EBNA3A)": "EBNA3",
    "BP1": "BP1",
    "5-(2-oxopropylideneamino)-6-D-ribitylaminouracil (5-OP-RU)": "5-OP-RU",
    "Latent membrane protein 1 (LMP1)": "LMP1",
    "Replication and transcription activator (BRLF1)": "BRLF1",
    "Trans activator protein (BZLF1),mRNA export factor ICP27 homolog (BMLF1)": "BZLF1",
    "ORF1ab,Non-structural protein 3 (NSP3)": "R1AB",
    "Ring finger protein 43-Mutant (RNF43-Mutant)": "RNF43mut",
    "ORF1ab": "R1AB",
    "Protein UL29/UL28": "UL29/28",
    "NY-ESO-1": "NY-ESO-1",
    "Regulatory protein E2": "E2",
}

### FLRGRAYGL was annotated as EBNA3A and not a combination of multiple antigens (EBNA3+BZLF1)
## RPPIFIRRL was also annotaed as EBNA3A manually 
## GLCTLVAML was annotated BMLF1
## RAKFKQLL was annotated BZLF1 only
## TTDPSFLGRY was annotated R1AB according to uniprot

## 
IMW_ann["IMW_antigen"] = IMW_ann["Epitope Antigen"].map(antigen_map)
IMW_ann["IMW_HLA"] = IMW_ann["Epitope MHC"]
IMW_ann["IMW_epitope"] = IMW_ann["Epitope"]
IMW_ann = IMW_ann.rename(columns={"clone_id" : "clone_handle"})


## Finalized
IMW_ann_final = IMW_ann[["clone_handle", "IMW_epitope", "IMW_HLA", "IMW_antigen"]].drop_duplicates()
print(IMW_ann_final.head(5))

## clone_list_primary
clone_list_primary = IMW_ann_final["clone_handle"].tolist()

# Annotate using ImmuneWatch predictions (secondary alpha chain)

In [ ]:
IMW_ann = pd.read_table(sep="\t", filepath_or_buffer="../../03_annotation/12_computational_agspecificity_annotation/Immunewatch_1.4.7/2027_7_PrimaryBeta_SecondaryAlpha_CD8T_mults_main_predictions.tsv")

#Cutoff suggestion for the Score is 0.2
IMW_ann = IMW_ann[IMW_ann["Score"] >= 0.2]

#select columns
IMW_ann = IMW_ann[["clone_id", "Epitope", "Epitope Antigen", "Epitope Species", "Epitope MHC"]]

# ## Translate epitope species 
antigen_map = {
    "Phosphoprotein 65-Mutant (pp65-Mutant)": "pp65",
    "Phosphoprotein 65 (pp65)": "pp65",
    "Matrix protein 1 (M1)": "M",  # influenza matrix protein
    "Epstein-Barr nuclear antigen 1 (EBNA1)": "EBNA1",
    "Spike/surface glycoprotein (S)": "S",
    "Epstein-Barr nuclear antigen 3A (EBNA3A),Epstein-Barr nuclear antigen 3 (EBNA3)": "EBNA3",
    "mRNA export factor ICP27 homolog (BMLF1),Replication and transcription activator (BRLF1)": "BMLF1",
    "NS3-Mutant,K1Y": "NS3mut",
    "p24 Gag": "p24",
    "Epstein-Barr nuclear antigen 3A (EBNA3A),Epstein-Barr nuclear antigen 3 (EBNA3),Trans activator protein (BZLF1)": "EBNA3",
    "Latent membrane protein 2A (LMP2A),Latent membrane protein 2 (LMP2)": "LMP2",
    "Gag,p24 Gag": "p24",
    "Transient Receptor Potential Cation Channel Subfamily V Member 4-Mutant (TRPV4-Mutant)": "TRPV4mut",
    "Non-structural protein 4b (NS4b),Polyprotein": "NS4b",
    "Splicing factor 3B subunit 1-Mutant (SF3B1-Mutant)": "SF3B1mut",
    "KRAS-Mutant": "KRASmut",
    "Nucleocapsid protein (N)": "N",
    "Trans activator protein (BZLF1)": "BZLF1",
    "Melanoma antigen recognized by T-cells 1 (MART1)": "MART1",
    "Epstein-Barr nuclear antigen 3A (EBNA3A)": "EBNA3",
    "BP1": "BP1",
    "5-(2-oxopropylideneamino)-6-D-ribitylaminouracil (5-OP-RU)": "5-OP-RU",
    "Latent membrane protein 1 (LMP1)": "LMP1",
    "Replication and transcription activator (BRLF1)": "BRLF1",
    "Trans activator protein (BZLF1),mRNA export factor ICP27 homolog (BMLF1)": "BZLF1",
    "ORF1ab,Non-structural protein 3 (NSP3)": "R1AB",
    "Ring finger protein 43-Mutant (RNF43-Mutant)": "RNF43mut",
    "ORF1ab": "R1AB",
    "Protein UL29/UL28": "UL29/28",
    "NY-ESO-1": "NY-ESO-1",
    "Regulatory protein E2": "E2",
}

#Checck whether keys correspond to the list
mapped_antigens = set(antigen_map.keys())
unique_antigens = set(IMW_ann["Epitope Antigen"].dropna().unique())
missing = sorted(unique_antigens - mapped_antigens)
print(f"{len(missing)} antigens are missing from the mapping:")
for x in missing:
    print(x)

# ## 
IMW_ann["IMW_antigen2"] = IMW_ann["Epitope Antigen"].map(antigen_map)
IMW_ann["IMW_HLA2"] = IMW_ann["Epitope MHC"]
IMW_ann["IMW_epitope2"] = IMW_ann["Epitope"]
IMW_ann = IMW_ann.rename(columns={"clone_id" : "clone_handle"})


# ## 
IMW_ann_final_2ndTRA = IMW_ann[["clone_handle", "IMW_epitope2", "IMW_HLA2", "IMW_antigen2"]].drop_duplicates()
IMW_ann_final_2ndTRA.head(5)

### **Did I gain any additional information from running the analysis with the secondary alpha chains?**

In [ ]:
## Dataframe sizes
print(len(IMW_ann_final))
print(len(IMW_ann_final_2ndTRA))  ## Secondary chain analysis contains fewer rows

## full join them
total_IMW = IMW_ann_final.merge(IMW_ann_final_2ndTRA, how="outer") 

## Number of chains annotated using primary chain information
print("I had primary TRA+TRB in one analysis, and secondary *TRA2* plus primary TRB chain in another run (TRA2+TRB)")
print("Number of annotated clonotypes in total: ", len(total_IMW))
print("Number of chains annotated using TRA+TRB *only*: ", len(total_IMW[total_IMW["IMW_epitope2"].isna()]))
print("Number of chains annotated using TRA2+TRB *only*: ", len(total_IMW[total_IMW["IMW_epitope"].isna()]))
print("Number of annotated clonotypes in both analysis (likely driven by TRB): ", len(total_IMW[(~total_IMW["IMW_epitope"].isna()) & (~total_IMW["IMW_epitope2"].isna())]))

## So yes, I did gain 22 more annotations from looking at secondary 

total_IMW["IMW_epitope"] = np.where(total_IMW["IMW_epitope"].isna(), total_IMW["IMW_epitope2"], total_IMW["IMW_epitope"])
total_IMW["IMW_HLA"] = np.where(total_IMW["IMW_HLA"].isna(), total_IMW["IMW_HLA2"], total_IMW["IMW_HLA"])
total_IMW["IMW_antigen"] = np.where(total_IMW["IMW_antigen"].isna(), total_IMW["IMW_antigen2"], total_IMW["IMW_antigen"])

### **Consolidate columns from IMW to one annotation**
total_IMW = total_IMW[["clone_handle", "IMW_epitope", "IMW_HLA", "IMW_antigen"]]

# Annotate using Stefan Schattgens data

In [ ]:
sschat_ann_known = pd.read_table(sep="\t", filepath_or_buffer="../../03_annotation/12_computational_agspecificity_annotation/PThomas_SSchattgen_EBV_TCRs/big_combo_tcrs_2024-02-02a_gp4_gex_w_tcr_obs_known_EBV_TCRs.tsv")
sschat_ann_known["Virus_SS_known"] = "EBV"
sschat_ann_known = sschat_ann_known.rename(columns={
                                           "cdr3a" : "IR_VJ_1_junction_aa" ,
                                           "cdr3b" : "IR_VDJ_1_junction_aa"})
sschat_ann_known = sschat_ann_known[["IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa", "Virus_SS_known"]].drop_duplicates()

sschat_ann_predicted = pd.read_table(sep="\t", filepath_or_buffer="../../03_annotation/12_computational_agspecificity_annotation/PThomas_SSchattgen_EBV_TCRs/big_combo_tcrs_2024-02-02a_gp4_gex_w_tcr_obs_predicted_EBV_TCRs.tsv")
sschat_ann_predicted["Virus_SS_pred"] = "EBV"
sschat_ann_predicted = sschat_ann_predicted.rename(columns={
                                           "cdr3a" : "IR_VJ_1_junction_aa" ,
                                           "cdr3b" : "IR_VDJ_1_junction_aa"})
sschat_ann_predicted = sschat_ann_predicted[["IR_VJ_1_junction_aa", "IR_VDJ_1_junction_aa", "Virus_SS_pred"]].drop_duplicates()
sschat_ann_predicted


# Annotate using VDJdb outputs + define a dataframe to merge annotations on to

In [ ]:
## This sit in the original obs_df file
obs_df = mdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[["clone_handle", 
                 'IR_VJ_1_v_call',
                  'IR_VJ_1_j_call',
                  'IR_VJ_1_junction_aa',
                 'IR_VDJ_1_v_call',
                 'IR_VDJ_1_d_call', 
                 'IR_VDJ_1_j_call',
                 "IR_VDJ_1_junction_aa",
                 "antigen.species", 
                 "antigen.gene",
                 "antigen.epitope"]].drop_duplicates()
obs_df.sort_values("antigen.epitope")

## Correct antigen.species
obs_df["antigen.species"].drop_duplicates().tolist()

virus_standardization = {
    "EBV": "EBV",
    "CMV": "CMV",
    "InfluenzaA": "FLU-A",
    "SARS-CoV-2": "SARS-COV-2",
    "ambiguous" : np.nan}

## Correct
obs_df["antigen.species"] = obs_df["antigen.species"].map(virus_standardization)

## Correct antigen.species
obs_df["antigen.gene"].drop_duplicates().tolist()
ag_standardization = {
    "LMP2A": "LMP2",
    "IE1": "IE1",
    "M": "M",
    "ORF14": "ORF14",
    "BMLF1" : "BMLF1", 
    "Spike": "S",
    "pp65": "pp65",
    "EBNA4": "EBNA4",
    "EBNA3": "EBNA3",
    "ambiguous": np.nan,
    "BZLF1": "BZLF1",
    "ORF1ab" : "ORF1ab"}
# obs_df["antigen.gene"] = obs_df["antigen.gene"].map(ag_standardization)

obs_df["antigen.gene"].drop_duplicates().tolist()

# Merge annotations from individual sources with keys from original dataframe

In [ ]:
## Multimer-based annotations
print(len(obs_df))
obs_df = obs_df.merge(mult_ann, how = "left")
print(len(obs_df))
print(obs_df.columns)

## Immunewatch detect
print(len(obs_df))
obs_df = obs_df.merge(total_IMW, how = "left")
print(len(obs_df))
print(obs_df.columns)

## Steffan Schattgen - known EBV TCRs
print(len(obs_df))
obs_df = obs_df.merge(sschat_ann_known, how = "left")
print(len(obs_df))
print(obs_df.columns)

## Steffan Schattgen - predicted EBV TCRs
print(len(obs_df))
obs_df = obs_df.merge(sschat_ann_predicted, how = "left")
print(len(obs_df))
print(obs_df.columns)



In [ ]:
obs_df.columns

# Complete Virus annotations

In [ ]:
IMW_antigen_to_virus = {
    np.nan: np.nan,
    "pp65": "CMV",
    "BMLF1": "EBV",
    "LMP2": "EBV",
    "M": "FLU-A",          # Influenza matrix protein
    "EBNA3": "EBV",
    "S": "SARS-COV-2",
    "BP1": "FLU-A",         # Unknown / synthetic / not virus-specific
    "5-OP-RU": "5-OP-RU",     # MR1 ligand, not viral
    "N": "SARS-COV-2",
    "R1AB": "SARS-COV-2",  # ORF1ab
    "EBNA1": "EBV",
    "BZLF1": "EBV",
    "BRLF1": "EBV",
    "MART1": "Cancer",       # Human melanoma antigen
    "RNF43mut": "Cancer",    # Tumor neoantigen
    "NY-ESO-1": "Cancer",    # Cancer-testis antigen
    "p24": "HIV",
    "NS3mut": "Cancer",      # Mutant antigen (virus not inferable)
    "NS4b": "YFV",         # Hepatitis C virus NS4B
    "KRASmut": "Cancer",     # Tumor neoantigen
    "E2": "HPV",           # HCV envelope glycoprotein E2
    "SF3B1mut": "Cancer",    # Tumor neoantigen
    "TRPV4mut": "Cancer",
    "UL29/28": "HHV-1",    # HSV-1 (UL29 = ICP8)
    "LMP1": "EBV",
}

## Annotate using IMV_antigen
obs_df["IMW_virus"] = obs_df["IMW_antigen"].map(IMW_antigen_to_virus)

In [ ]:
## Ensure the same names are used for virus throughout
final_df = obs_df[["clone_handle", "antigen.species", "antigen.gene", "Virus", "Antigen", "IMW_epitope", "IMW_antigen", "IMW_virus", "Virus_SS_known", "Virus_SS_pred"]]

#Drop colunns that are NaN in all columns except clone_handle
cols_to_check = final_df.columns.difference(["clone_handle"])
final_df = final_df.dropna(subset=cols_to_check, how="all")


# Upset plot with viruses as input

In [ ]:
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_indicators

annot_cols = [
    "antigen.species",
    "Virus",
    "IMW_virus",
    "Virus_SS_known",
    "Virus_SS_pred",
]

df_bin = final_df.set_index("clone_handle")[annot_cols].notna()

upset_data = from_indicators(df_bin.columns, df_bin)

fig = plt.figure(figsize=(10, 5))

UpSet(
    upset_data,
    show_counts=True,
    sort_by="cardinality",
).plot(fig=fig)

fig.tight_layout()

fig.savefig(
    "../../05_plots/09_multimers/45_Upset_plot_annotations.pdf",
    format="pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

### Kilian wants to know how often IMW predicts  the same thing as our multimers

In [ ]:
test_df = final_df[(~final_df["Virus"].isna()) & (~final_df["IMW_virus"].isna())]
test_df["agreement"] = np.where(test_df["Virus"]==test_df["IMW_virus"], "Same pathogen", "Different pathogens")

print(test_df["agreement"].value_counts())

#Which one was different?
clone_lookup = test_df[test_df["Virus"] != test_df["IMW_virus"]]["clone_handle"].tolist()
print(clone_lookup)

## Cell counts and annotations with our multimers
df = mdata["gex"].obs[mdata["gex"].obs["clone_handle"].isin(clone_lookup)][["clone_handle", "Compartment", "type_x_clone_count"]].drop_duplicates()
print(df.merge(mult_ann, how="left"))

## annotations by immune watch
IMW_ann[IMW_ann["clone_handle"].isin(clone_lookup)]

## Can we generally display the number of times the 4 different work flows disagree?

In [ ]:
## Virus and antigen species
test_df = final_df[["antigen.species", "Virus"]].dropna()
n_equal = (test_df["antigen.species"] == test_df["Virus"]).sum()
n_not_equal = (test_df["antigen.species"] != test_df["Virus"]).sum()

print(f"Equal: {n_equal}")
print(f"Not equal: {n_not_equal}")

In [ ]:
## Virus and antigen species
test_df = final_df[["Virus_SS_known", "Virus"]].dropna()
n_equal = (test_df["Virus_SS_known"] == test_df["Virus"]).sum()
n_not_equal = (test_df["Virus_SS_known"] != test_df["Virus"]).sum()

print(f"Equal: {n_equal}")
print(f"Not equal: {n_not_equal}")

In [ ]:
## Virus and antigen species
test_df = final_df[["Virus_SS_pred", "Virus"]].dropna()
n_equal = (test_df["Virus_SS_pred"] == test_df["Virus"]).sum()
n_not_equal = (test_df["Virus_SS_pred"] != test_df["Virus"]).sum()

print(f"Equal: {n_equal}")
print(f"Not equal: {n_not_equal}")

In [ ]:
final_df.columns

In [ ]:
final_df[final_df["clone_handle"] == "Pt8-3399"]

# Export

In [ ]:
#prioritize annotations
## 1st multimers
## 2nd IMW_virus
## 3rd Virus_SS_known
## 4th antigen.species

final_df["Virus_final"] = np.where(~final_df["Virus"].isna(), final_df["Virus"], 
                                   np.where(~final_df["IMW_virus"].isna(), final_df["IMW_virus"], 
                                            np.where(~final_df["Virus_SS_known"].isna(), final_df["Virus_SS_known"], 
                                                      np.where(~final_df["antigen.species"].isna(), final_df["antigen.species"], np.nan))))

final_df["Antigen_final"] = np.where((final_df["Antigen"].isna()) & (final_df["IMW_antigen"].isna()), final_df["antigen.gene"], 
                                np.where(final_df["IMW_antigen"].notna(), final_df["IMW_antigen"], final_df["Antigen"]))

## UL28/29 is from the FRCPRRFCF epitope, which we excluded
final_df["Antigen_final"] = np.where(final_df["Antigen"] == "UL28/29", np.nan, final_df["Antigen_final"])

# Consolidate
Total_annotations = final_df[["clone_handle", "Virus_final", "Antigen_final"]]
Total_annotations.to_csv("../../06_csv_outputs/Clones/2026_7_total_VirusAnnotationsPerClone.csv")
print(len(Total_annotations))
print(len(mult_ann))

In [ ]:
final_df[final_df["clone_handle"] == "Pt8-3399"]

In [ ]:
final_df[(final_df["Antigen"].isna()) & (~final_df["IMW_antigen"].isna())]